# Module 3: Model Training with Snowpark ML

**Snowpark ML Fundamentals - Week 1 | Lunch & Learn**

---

## Learning Objectives
- Train ML models using `snowflake.ml.modeling` (scikit-learn compatible API)
- Compare XGBoost, RandomForest, and LogisticRegression
- Evaluate models with distributed metrics
- Understand feature importance

## Key Concept
> **Snowpark ML wraps scikit-learn, XGBoost, and LightGBM** with the same API.
> Training happens inside the Snowflake warehouse using `input_cols`, `label_cols`,
> and `output_cols` parameters instead of numpy arrays.

---

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3.1 Setup & Prepare Data

In [21]:
import sys
sys.path.insert(0, '../src')

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline

session = create_session(env_path='../.env')

# Create dataset
df = create_customer_churn_dataset(session, n_rows=5000)

# Define features
NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'

# Preprocess
df_processed, _ = build_preprocessing_pipeline(
    df, NUMERIC_COLS, CATEGORICAL_COLS,
    numeric_method='standard', categorical_method='ordinal'
)

# Train/Test split (80/20)
train_df, test_df = split_data(df_processed, train_ratio=0.8)
print(f"Training set: {train_df.count()} rows")
print(f"Test set:     {test_df.count()} rows")

Training set: 3962 rows
Test set:     1038 rows


In [22]:
train_df.show()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |
----------------------------------------------------------------------------

## 3.2 Train an XGBoost Classifier

Snowpark ML's `XGBClassifier` wraps the XGBoost library with the same parameters
but operates on Snowpark DataFrames.

In [23]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.modeling.trainer import train_model, predict

# Feature columns after preprocessing
FEATURE_COLS = (
    [f"{c}_SCALED" for c in NUMERIC_COLS] +
    [f"{c}_ENCODED" for c in CATEGORICAL_COLS]
)

# Train XGBoost
xgb_model = train_model(
    train_df=train_df,
    feature_cols=FEATURE_COLS,
    label_col=LABEL_COL,
    model_type='xgboost',
    model_params={
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
    },
)
print("XGBoost model trained successfully!")

The version of package 'snowflake-snowpark-python' in the local environment is 1.45.0, which does not fit the criteria for the requirement 'snowflake-snowpark-python'. Your UDF might not work when the package version is different between the server and your local environment.
The version of package 'snowflake-telemetry-python' in the local environment is 0.7.1, which does not fit the criteria for the requirement 'snowflake-telemetry-python'. Your UDF might not work when the package version is different between the server and your local environment.


XGBoost model trained successfully!


In [24]:
# Generate predictions
predictions = predict(xgb_model, test_df)
predictions.select(LABEL_COL, 'PREDICTION').show(10)

----------------------------
|"CHURNED"  |"PREDICTION"  |
----------------------------
|0          |0             |
|0          |0             |
|1          |1             |
|0          |0             |
|0          |0             |
|1          |1             |
|1          |1             |
|0          |0             |
|1          |1             |
|0          |0             |
----------------------------



## 3.3 Evaluate the Model

In [14]:
from snowpark_fundamentals.modeling.evaluation import (
    evaluate_binary_classifier,
    get_confusion_matrix,
    get_feature_importance,
)

# Metrics
metrics = evaluate_binary_classifier(predictions, LABEL_COL, 'PREDICTION')
print("XGBoost Performance:")
for metric, value in metrics.items():
    print(f"  {metric:>12}: {value:.4f}")

XGBoost Performance:
      accuracy: 0.8410
     precision: 0.9036
        recall: 0.3233
      f1_score: 0.4762


In [15]:
# Confusion Matrix
cm = get_confusion_matrix(predictions, LABEL_COL, 'PREDICTION')
print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[798.   8.]
 [157.  75.]]


In [25]:
# Feature Importance
importance = get_feature_importance(xgb_model, FEATURE_COLS)
print("\nTop 10 Features:")
for feat in importance[:10]:
    print(f"  {feat['feature']:40} {feat['importance']:.4f}")


Top 10 Features:
  SUPPORT_TICKETS_SCALED                   0.2912
  USAGE_HOURS_PER_WEEK_SCALED              0.1357
  CONTRACT_TYPE_ENCODED                    0.1124
  TENURE_MONTHS_SCALED                     0.0954
  MONTHLY_CHARGES_SCALED                   0.0700
  TOTAL_CHARGES_SCALED                     0.0658
  PAYMENT_METHOD_ENCODED                   0.0653
  NUM_DEPENDENTS_SCALED                    0.0586
  AGE_SCALED                               0.0560
  PLAN_TYPE_ENCODED                        0.0497


## 3.4 Compare Multiple Models

In [17]:
# Train RandomForest
rf_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='random_forest',
    model_params={'n_estimators': 100, 'max_depth': 10},
)
rf_preds = predict(rf_model, test_df)
rf_metrics = evaluate_binary_classifier(rf_preds, LABEL_COL, 'PREDICTION')

# Train Logistic Regression
lr_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='logistic_regression',
    model_params={'max_iter': 1000},
)
lr_preds = predict(lr_model, test_df)
lr_metrics = evaluate_binary_classifier(lr_preds, LABEL_COL, 'PREDICTION')

# Comparison table
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 65)
for name, m in [('XGBoost', metrics), ('Random Forest', rf_metrics), ('Logistic Regression', lr_metrics)]:
    print(f"{name:<25} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1_score']:>10.4f}")

The version of package 'snowflake-snowpark-python' in the local environment is 1.45.0, which does not fit the criteria for the requirement 'snowflake-snowpark-python'. Your UDF might not work when the package version is different between the server and your local environment.
The version of package 'snowflake-telemetry-python' in the local environment is 0.7.1, which does not fit the criteria for the requirement 'snowflake-telemetry-python'. Your UDF might not work when the package version is different between the server and your local environment.


Model                       Accuracy  Precision     Recall         F1
-----------------------------------------------------------------
XGBoost                       0.8410     0.9036     0.3233     0.4762
Random Forest                 0.8439     0.9730     0.3103     0.4706
Logistic Regression           0.7900     0.8889     0.0690     0.1280


## 3.5 Save Results to Snowflake

Persist predictions and model comparison to tables so results are queryable in the Snowflake UI.

In [26]:
predictions.show()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |"PREDICTION"  |
----------------------------------------------

In [18]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType, FloatType, StructType, StructField

# 1. Save XGBoost predictions to a table
predictions.select(
    'CUSTOMER_ID', 'CHURNED', 'PREDICTION'
).write.save_as_table('CHURN_PREDICTIONS_XGBOOST', mode='overwrite')
print(f"Saved {predictions.count()} predictions to CHURN_PREDICTIONS_XGBOOST")

# 2. Save model comparison results to a table
comparison_data = []
for name, m in [('XGBoost', metrics), ('Random Forest', rf_metrics), ('Logistic Regression', lr_metrics)]:
    comparison_data.append([name, m['accuracy'], m['precision'], m['recall'], m['f1_score']])

comparison_df = session.create_dataframe(
    comparison_data,
    schema=['MODEL_NAME', 'ACCURACY', 'PRECISION', 'RECALL', 'F1_SCORE']
)
comparison_df.write.save_as_table('MODEL_COMPARISON_RESULTS', mode='overwrite')
print("Saved model comparison to MODEL_COMPARISON_RESULTS")

# 3. Save feature importance to a table
importance_data = [[feat['feature'], feat['importance']] for feat in importance]
importance_df = session.create_dataframe(importance_data, schema=['FEATURE_NAME', 'IMPORTANCE'])
importance_df.write.save_as_table('FEATURE_IMPORTANCE_XGBOOST', mode='overwrite')
print("Saved feature importance to FEATURE_IMPORTANCE_XGBOOST")

# Verify
session.table('MODEL_COMPARISON_RESULTS').show()

Saved 1038 predictions to CHURN_PREDICTIONS_XGBOOST
Saved model comparison to MODEL_COMPARISON_RESULTS
Saved feature importance to FEATURE_IMPORTANCE_XGBOOST
----------------------------------------------------------------------------------------------------
|"MODEL_NAME"         |"ACCURACY"  |"PRECISION"         |"RECALL"             |"F1_SCORE"          |
----------------------------------------------------------------------------------------------------
|XGBoost              |0.84104     |0.9036144578313253  |0.3232758620689655   |0.4761904761904763  |
|Random Forest        |0.843931    |0.972972972972973   |0.3103448275862069   |0.4705882352941177  |
|Logistic Regression  |0.789981    |0.8888888888888888  |0.06896551724137931  |0.128               |
----------------------------------------------------------------------------------------------------



## Key Takeaways

1. `snowflake.ml.modeling` provides **scikit-learn compatible wrappers** for XGBoost, RandomForest, LogisticRegression, and 30+ other algorithms
2. Use `input_cols`, `label_cols`, `output_cols` instead of numpy arrays
3. Training runs **inside the Snowflake warehouse** (distributed compute)
4. Evaluation metrics are computed **server-side** using `snowflake.ml.modeling.metrics`
5. Feature importance helps interpret which features drive predictions

---

**Next: [04 - Model Registry](04_model_registry.ipynb)**

In [19]:
session.close()